<a href="https://colab.research.google.com/github/youngbean0519/Self-Project/blob/main/notebooks/11_koelectra.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path
import subprocess
import sys

PROJECT_DIR = Path("/content/Self-Project")
SRC_DIR = PROJECT_DIR / "src"

print("Python:", sys.version)

if not PROJECT_DIR.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/youngbean0519/Self-Project.git",
            str(PROJECT_DIR),
        ],
        check=True,
    )

# tokenizers는 소스 컴파일 대신 빌드된 wheel 사용
subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--upgrade",
        "--only-binary=tokenizers",
        "transformers==4.57.6",
        "tokenizers>=0.22.0,<=0.23.0",
        "sentencepiece",
        "kiwipiepy",
    ],
    check=True,
)

for path in [PROJECT_DIR, SRC_DIR]:
    if str(path) not in sys.path:
        sys.path.append(str(path))

print("설치 완료")

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
설치 완료


In [ ]:
import copy
import gc
import hashlib
import json
import math
import random
import time
from contextlib import nullcontext

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import transformers

from IPython.display import display
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    hamming_loss,
    precision_score,
    recall_score,
)
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from transformers import ElectraModel, get_linear_schedule_with_warmup
from transformers import AutoTokenizer

import preprocessing.text as text_preprocessing
import utils.labels as utils_labels

DATA_DIR = Path("/content/drive/MyDrive/26_KOTE/data")
ARTIFACT_DIR = Path("/content/drive/MyDrive/26_KOTE/artifacts/week11_koelectra")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

LABEL_NAMES = list(utils_labels.LABELS)
NUM_LABELS = 44
MODEL_NAME = "monologg/koelectra-base-v3-discriminator"

assert len(LABEL_NAMES) == NUM_LABELS
assert DEVICE.type == "cuda", "Colab 런타임을 GPU로 변경해 주세요."

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("결과 저장:", ARTIFACT_DIR)

def set_seed(seed):
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)

  torch.backends.cudnn.deterministic = True
  torch.backends.cudnn.benchmark = False

def save_json(data, path):
  with open(path, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

def config_id(config):
  text = json.dumps(config, sort_keys=True)
  return hashlib.sha256(text.encode()).hexdigest()[:12]

def synchronize():
  torch.cuda.synchronize()

def autocast_context():
  if DEVICE.type == "cuda":
    return torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
    )
  return nullcontext()
save_json(LABEL_NAMES, ARTIFACT_DIR / "label_names.json")
print("레이블:", list(enumerate(LABEL_NAMES)))


PyTorch: 2.14.1+cu130
Transformers: 4.57.6
GPU: NVIDIA A100-SXM4-40GB
결과 저장: /content/drive/MyDrive/26_KOTE/artifacts/week11_koelectra
레이블: [(0, '불평/불만'), (1, '환영/호의'), (2, '감동/감탄'), (3, '지긋지긋'), (4, '고마움'), (5, '슬픔'), (6, '화남/분노'), (7, '존경'), (8, '기대감'), (9, '우쭐댐/무시함'), (10, '안타까움/실망'), (11, '비장함'), (12, '의심/불신'), (13, '뿌듯함'), (14, '편안/쾌적'), (15, '신기함/관심'), (16, '아껴주는'), (17, '부끄러움'), (18, '공포/무서움'), (19, '절망'), (20, '한심함'), (21, '역겨움/징그러움'), (22, '짜증'), (23, '어이없음'), (24, '없음'), (25, '패배/자기혐오'), (26, '귀찮음'), (27, '힘듦/지침'), (28, '즐거움/신남'), (29, '깨달음'), (30, '죄책감'), (31, '증오/혐오'), (32, '흐뭇함(귀여움/예쁨)'), (33, '당황/난처'), (34, '경악'), (35, '부담/안_내킴'), (36, '서러움'), (37, '재미없음'), (38, '불쌍함/연민'), (39, '놀람'), (40, '행복'), (41, '불안/걱정'), (42, '기쁨'), (43, '안심/신뢰')]


In [ ]:
def load_split(name):
    return pd.read_csv(
        DATA_DIR / f"{name}.tsv",
        sep="\t",
        header=None,
        names=["id", "text", "labels"],
    )


train_df = load_split("train")
val_df = load_split("val")


def make_labels(df):
    targets = np.zeros(
        (len(df), NUM_LABELS),
        dtype=np.float32,
    )

    for i, value in enumerate(df["labels"].astype(str)):
        indices = [
            int(x.strip())
            for x in value.split(",")
            if x.strip()
        ]
        assert indices and all(0 <= j < NUM_LABELS for j in indices), (i, value)
        targets[i, indices] = 1.0

    return targets


y_train = make_labels(train_df)
y_val = make_labels(val_df)

PREPROCESSING = {
    "repeat": {
        "normalize_repeat": True,
        "remove_symbols": False,
    },
    "minimal": {
        "normalize_repeat": False,
        "remove_symbols": False,
    },
    "repeat_symbols": {
        "normalize_repeat": True,
        "remove_symbols": True,
    },
}

TEXT_CACHE = {}


def get_texts(mode):
    if mode not in TEXT_CACHE:
        kwargs = PREPROCESSING[mode]

        def convert(df):
            return (
                df["text"]
                .fillna("")
                .astype(str)
                .apply(
                    lambda text: text_preprocessing.preprocess_text(
                        text, **kwargs
                    )
                )
                .tolist()
            )

        TEXT_CACHE[mode] = (
            convert(train_df),
            convert(val_df),
        )

    return TEXT_CACHE[mode]


print("Train:", train_df.shape, y_train.shape)
print("Validation:", val_df.shape, y_val.shape)

assert y_train.shape == (40000, NUM_LABELS)
assert y_val.shape == (5000, NUM_LABELS)
assert train_df["id"].is_unique and val_df["id"].is_unique
assert set(train_df["id"]).isdisjoint(set(val_df["id"]))


Train: (40000, 3) (40000, 44)
Validation: (5000, 3) (5000, 44)


## KoELECTRA 토크나이저 및 Dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
tokenizer.padding_side = "right"
tokenizer.truncation_side ="right"

example_ids = tokenizer.encode("한국어 모델을 공유합니다.")
print("예제 토큰 ID:", example_ids)

assert example_ids[0] == tokenizer.cls_token_id
assert example_ids[-1] == tokenizer.sep_token_id
print("예제 토큰:", tokenizer.convert_ids_to_tokens(example_ids))
tokenizer.save_pretrained(ARTIFACT_DIR / "tokenizer")

INPUT_CASE = {}

def encode_texts(texts, max_len):
  return tokenizer(
      texts,
      padding="max_length",
      truncation=True,
      max_length=max_len,
      return_attention_mask=True,
      return_token_type_ids=False,
      return_tensors="pt",
  )

def get_encoded_inputs(config):
  key = (
      config["preprocessing"],
      config["max_len"],
  )

  if key not in INPUT_CASE:
    train_texts, val_texts = get_texts(config["preprocessing"])

    INPUT_CASE[key] = (
        encode_texts(train_texts, config["max_len"]),
        encode_texts(val_texts, config["max_len"]),
    )

  return INPUT_CASE[key]

def input_statistics(texts, max_len):
  encoded = tokenizer(
      texts,
      add_special_tokens=True,
      truncation=False,
      padding=False,
      return_attention_mask=False,
      return_token_type_ids=False,
  )["input_ids"]

  lengths = np.array([len(ids) for ids in encoded])
  total_tokens = sum(len(ids) for ids in encoded)
  unknown_tokens = sum(ids.count(tokenizer.unk_token_id) for ids in encoded)

  return {
      "mean_tokens": float(lengths.mean()),
      "p95": float(np.percentile(lengths, 95)),
      "p99": float(np.percentile(lengths, 99)),
      "truncation_rate": float((lengths > max_len).mean()),
      "unk_ratio": unknown_tokens / max(total_tokens, 1),
  }

class KoELECTRADataset(Dataset):
  def __init__(self, encodings, labels):
    self.encodings = encodings
    self.labels = torch.as_tensor(labels, dtype=torch.float32)

  def __len__(self):
    return len(self.labels)

  def __getitem__(self, index):
    return {
        "input_ids": self.encodings["input_ids"][index],
        "attention_mask": self.encodings["attention_mask"][index],
        "labels": self.labels[index],
    }

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/61.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/467 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

예제 토큰 ID: [2, 11229, 6918, 4292, 7824, 17788, 18, 3]
예제 토큰: ['[CLS]', '한국어', '모델', '##을', '공유', '##합니다', '.', '[SEP]']


## 분류 모델 및 공통 초기 설정

In [ ]:
class KoELECTRAClassifier(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.electra = ElectraModel.from_pretrained(
            config["model_name"]
        )
        self.dropout = nn.Dropout(config["dropout"])
        # KoBERT의 pooler에 대응하는 학습 가능한 문장 변환층
        self.dense = nn.Linear(self.electra.config.hidden_size, self.electra.config.hidden_size)
        self.classifier = nn.Linear(
            self.electra.config.hidden_size,
            NUM_LABELS,
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.electra(
            input_ids=input_ids,
            attention_mask=attention_mask,
            # 단일 문장이므로 token_type_ids는 기본값 0 사용
            return_dict=True,
        )

        features = torch.tanh(self.dense(outputs.last_hidden_state[:, 0, :]))
        features = self.dropout(features)
        return self.classifier(features)


BASE_CONFIG = {
    "implementation_version": 2,
    "head": "cls_dense_tanh_dropout_linear",
    "model_name": MODEL_NAME,
    "seed": 42,
    "preprocessing": "repeat",
    "max_len": 128,
    "dropout": 0.1,
    "lr": 2e-5,
    "weight_decay": 0.01,
    "batch_size": 16,
    "max_epochs": 5,
    "patience": 2,
    "warmup_ratio": 0.1,
    "loss": "bce",
}

train_texts, val_texts = get_texts("repeat")

initial_stats = {
    "train": input_statistics(train_texts, 128),
    "validation": input_statistics(val_texts, 128),
}

display(pd.DataFrame(initial_stats).T)
save_json(
    initial_stats,
    ARTIFACT_DIR / "initial_input_statistics.json",
)

,mean_tokens,p95,p99,truncation_rate,unk_ratio
train,34.013625,81.0,137.00,0.013225,0.001363
validation,34.786600,82.0,134.01,0.012000,0.001271


## 평가 임계값 탐색 함수

In [ ]:
THRESHOLDS = np.round(
    np.arange(0.10, 0.61, 0.05),
    2,
)


def classification_metrics(labels, probabilities, threshold):
    predictions = (
        probabilities >= threshold
    ).astype(np.int32)

    return {
        "micro_f1": float(f1_score(
            labels, predictions,
            average="micro",
            zero_division=0,
        )),
        "macro_f1": float(f1_score(
            labels, predictions,
            average="macro",
            zero_division=0,
        )),
        "hamming_loss": float(hamming_loss(
            labels, predictions
        )),
        "precision_micro": float(precision_score(
            labels, predictions,
            average="micro",
            zero_division=0,
        )),
        "recall_micro": float(recall_score(
            labels, predictions,
            average="micro",
            zero_division=0,
        )),
        "predicted_labels_per_sample": float(
            predictions.sum(axis=1).mean()
        ),
    }


def threshold_search(
    labels,
    probabilities,
    thresholds=THRESHOLDS,
):
    table = pd.DataFrame([
        {
            "threshold": float(threshold),
            **classification_metrics(
                labels, probabilities, threshold
            ),
        }
        for threshold in thresholds
    ])

    best = table.loc[
        table["micro_f1"].idxmax()
    ].to_dict()

    return table, best


def labelwise_table(labels, probabilities, threshold):
    predictions = (
        probabilities >= threshold
    ).astype(np.int32)

    return pd.DataFrame({
        "label_id": np.arange(NUM_LABELS),
        "label": LABEL_NAMES,
        "support": labels.sum(axis=0).astype(int),
        "predicted": predictions.sum(axis=0),
        "precision": precision_score(
            labels, predictions,
            average=None, zero_division=0,
        ),
        "recall": recall_score(
            labels, predictions,
            average=None, zero_division=0,
        ),
        "f1": f1_score(
            labels, predictions,
            average=None, zero_division=0,
        ),
    })


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()

    total_loss = 0.0
    count = 0
    all_probabilities = []
    all_labels = []

    for batch in loader:
        input_ids = batch["input_ids"].to(
            DEVICE, non_blocking=True
        )
        attention_mask = batch["attention_mask"].to(
            DEVICE, non_blocking=True
        )
        labels = batch["labels"].to(
            DEVICE, non_blocking=True
        )

        with autocast_context():
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)

        total_loss += loss.item() * len(labels)
        count += len(labels)

        all_probabilities.append(
            torch.sigmoid(logits.float()).cpu().numpy()
        )
        all_labels.append(labels.cpu().numpy())

    return (
        total_loss / count,
        np.vstack(all_probabilities),
        np.vstack(all_labels),
    )

## 소규모 순전파 역전파 확인

In [ ]:
set_seed(42)

train_encodings, _ = get_encoded_inputs(BASE_CONFIG)

smoke_dataset = KoELECTRADataset(
    {
        key: value[:4]
        for key, value in train_encodings.items()
    },
    y_train[:4],
)

smoke_loader = DataLoader(
    smoke_dataset,
    batch_size=4,
)

smoke_model = KoELECTRAClassifier(BASE_CONFIG).to(DEVICE)
smoke_model.train()

batch = next(iter(smoke_loader))

with autocast_context():
  logits = smoke_model(
      batch["input_ids"].to(DEVICE),
      batch["attention_mask"].to(DEVICE),
  )

  loss = nn.BCEWithLogitsLoss()(
      logits,
      batch["labels"].to(DEVICE),
  )

assert logits.shape == (4, NUM_LABELS)
assert torch.isfinite(loss)
loss.backward()
assert smoke_model.classifier.weight.grad is not None
assert torch.isfinite(smoke_model.classifier.weight.grad).all()
print("동작 확인 완료:", tuple(logits.shape), "loss:", loss.item())
del smoke_model, smoke_loader, smoke_dataset, batch, logits, loss
gc.collect()
torch.cuda.empty_cache()


pytorch_model.bin:   0%|          | 0.00/452M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/452M [00:00<?, ?B/s]

동작 확인 완료: (4, 44) loss: 0.6951552629470825


## 학습 및 후보 비교 함

In [ ]:
def run_experiment(config, force=False):
    config = copy.deepcopy(config)
    run_id = config_id(config)

    run_dir = ARTIFACT_DIR / "runs" / run_id
    run_dir.mkdir(parents=True, exist_ok=True)

    summary_path = run_dir / "summary.json"
    checkpoint_path = run_dir / "best.pt"

    if summary_path.exists() and checkpoint_path.exists() and not force:
        print("저장된 결과 사용:", run_id)
        return json.loads(
            summary_path.read_text(encoding="utf-8")
        )

    train_encodings, val_encodings = get_encoded_inputs(config)

    train_dataset = KoELECTRADataset(
        train_encodings, y_train
    )
    val_dataset = KoELECTRADataset(
        val_encodings, y_val
    )

    set_seed(config["seed"])

    generator = torch.Generator().manual_seed(
        config["seed"]
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=config["batch_size"],
        shuffle=True,
        generator=generator,
        num_workers=0,
        pin_memory=True,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=config["batch_size"],
        shuffle=False,
        num_workers=0,
        pin_memory=True,
    )

    model = KoELECTRAClassifier(config).to(DEVICE)

    if config["loss"] == "weighted_bce":
        positives = y_train.sum(axis=0)
        negatives = len(y_train) - positives

        weights = np.minimum(
            5.0,
            np.sqrt(
                negatives / np.maximum(positives, 1)
            ),
        )

        criterion = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor(
                weights,
                dtype=torch.float32,
                device=DEVICE,
            )
        )
    else:
        criterion = nn.BCEWithLogitsLoss()

    # bias와 LayerNorm 파라미터에는 weight decay 제외
    decay_params = []
    no_decay_params = []

    for name, parameter in model.named_parameters():
        if "bias" in name or "LayerNorm.weight" in name:
            no_decay_params.append(parameter)
        else:
            decay_params.append(parameter)

    optimizer = torch.optim.AdamW(
        [
            {
                "params": decay_params,
                "weight_decay": config["weight_decay"],
            },
            {
                "params": no_decay_params,
                "weight_decay": 0.0,
            },
        ],
        lr=config["lr"],
    )

    total_steps = (
        len(train_loader) * config["max_epochs"]
    )

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(
            total_steps * config["warmup_ratio"]
        ),
        num_training_steps=total_steps,
    )

    scaler = torch.amp.GradScaler("cuda")

    best_score = -1.0
    best_epoch = 0
    patience_counter = 0
    history = []

    save_json(config, run_dir / "config.json")
    save_json(LABEL_NAMES, run_dir / "label_names.json")
    model.electra.config.save_pretrained(run_dir / "encoder_config")

    synchronize()
    start = time.perf_counter()

    for epoch in range(1, config["max_epochs"] + 1):
        model.train()
        total_loss = 0.0
        count = 0

        progress = tqdm(
            train_loader,
            desc=f"{run_id} | epoch {epoch}",
            leave=False,
        )

        for batch in progress:
            input_ids = batch["input_ids"].to(
                DEVICE, non_blocking=True
            )
            attention_mask = batch["attention_mask"].to(
                DEVICE, non_blocking=True
            )
            labels = batch["labels"].to(
                DEVICE, non_blocking=True
            )

            optimizer.zero_grad(set_to_none=True)

            with autocast_context():
                logits = model(
                    input_ids, attention_mask
                )
                loss = criterion(logits, labels)

            if not torch.isfinite(loss):
                raise RuntimeError(
                    "Loss에 NaN 또는 Inf가 발생했습니다."
                )

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)

            nn.utils.clip_grad_norm_(
                model.parameters(), 1.0
            )

            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()

            # overflow로 optimizer step이 생략되면 scheduler도 생략
            if scaler.get_scale() >= old_scale:
                scheduler.step()

            total_loss += loss.item() * len(labels)
            count += len(labels)

            progress.set_postfix(
                loss=f"{total_loss / count:.4f}"
            )

        val_loss, probabilities, labels = evaluate(
            model, val_loader, criterion
        )
        _, best = threshold_search(
            labels, probabilities
        )

        record = {
            "epoch": epoch,
            "train_loss": total_loss / count,
            "val_loss": val_loss,
            **best,
        }
        history.append(record)

        print(
            f"{run_id} | epoch {epoch:02d} | "
            f"train {record['train_loss']:.4f} | "
            f"val {val_loss:.4f} | "
            f"micro {best['micro_f1']:.4f} | "
            f"macro {best['macro_f1']:.4f} | "
            f"threshold {best['threshold']:.2f}"
        )

        pd.DataFrame(history).to_csv(
            run_dir / "history.csv",
            index=False,
        )

        if best["micro_f1"] > best_score:
            best_score = best["micro_f1"]
            best_epoch = epoch
            patience_counter = 0

            torch.save(
                {
                    key: value.detach().cpu()
                    for key, value in model.state_dict().items()
                },
                checkpoint_path,
            )
        else:
            patience_counter += 1

            if patience_counter >= config["patience"]:
                print("조기 종료")
                break

    synchronize()
    elapsed = time.perf_counter() - start

    state = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True,
    )
    model.load_state_dict(state)
    del state

    val_loss, probabilities, labels = evaluate(
        model, val_loader, criterion
    )
    threshold_table, best = threshold_search(
        labels, probabilities
    )

    threshold_table.to_csv(
        run_dir / "thresholds.csv", index=False
    )

    labelwise_table(
        labels, probabilities, best["threshold"]
    ).to_csv(
        run_dir / "labelwise.csv",
        index=False,
        encoding="utf-8-sig",
    )

    np.savez_compressed(
        run_dir / "validation_predictions.npz",
        probabilities=probabilities,
        labels=labels,
    )

    summary = {
        "run_id": run_id,
        "config": config,
        **best,
        "ap_micro": float(average_precision_score(
            labels, probabilities, average="micro"
        )),
        "ap_macro": float(average_precision_score(
            labels, probabilities, average="macro"
        )),
        "val_loss": float(val_loss),
        "best_epoch": best_epoch,
        "epochs_run": len(history),
        "training_seconds": elapsed,
        "parameters": sum(
            p.numel() for p in model.parameters()
        ),
        "gpu": torch.cuda.get_device_name(0),
        "torch_version": str(torch.__version__),
        "transformers_version": transformers.__version__,
        "test_evaluated": False,
    }

    save_json(summary, summary_path)

    del model, optimizer, scheduler, scaler, criterion
    gc.collect()
    torch.cuda.empty_cache()

    return summary


def result_table(results):
    rows = []

    for result in results:
        rows.append({
            "run_id": result["run_id"],
            **result["config"],
            **{
                key: result[key]
                for key in [
                    "micro_f1",
                    "macro_f1",
                    "hamming_loss",
                    "ap_micro",
                    "ap_macro",
                    "threshold",
                    "best_epoch",
                    "epochs_run",
                    "training_seconds",
                ]
            },
        })

    return (
        pd.DataFrame(rows)
        .sort_values("micro_f1", ascending=False)
        .reset_index(drop=True)
    )


def compare_candidates(base, changes, stage):
    configs = [copy.deepcopy(base)]

    for change in changes:
        candidate = copy.deepcopy(base)
        candidate.update(change)
        configs.append(candidate)

    configs = {
        config_id(config): config
        for config in configs
    }

    results = [
        run_experiment(config)
        for config in configs.values()
    ]

    table = result_table(results)

    print("\nStage:", stage)
    display(table)

    table.to_csv(
        ARTIFACT_DIR / f"{stage}.csv",
        index=False,
        encoding="utf-8-sig",
    )

    winner = max(
        results,
        key=lambda result: result["micro_f1"],
    )

    save_json(
        winner["config"],
        ARTIFACT_DIR / f"{stage}_winner.json",
    )

    return copy.deepcopy(winner["config"]), results

## 기준 실험

In [ ]:
baseline_result = run_experiment(BASE_CONFIG)
CURRENT = copy.deepcopy(BASE_CONFIG)

display(result_table([baseline_result]))

baseline_dir = (
    ARTIFACT_DIR / "runs" / baseline_result["run_id"]
)

history = pd.read_csv(
    baseline_dir / "history.csv"
)
display(history)

baseline_labels = pd.read_csv(
    baseline_dir / "labelwise.csv"
)

print("성능이 낮은 감정")
display(baseline_labels.sort_values("f1").head(10))

d9cf849de83a | epoch 1:   0%|          | 0/2500 [00:00<?, ?it/s]

d9cf849de83a | epoch 01 | train 0.3960 | val 0.3185 | micro 0.5961 | macro 0.4408 | threshold 0.30


d9cf849de83a | epoch 2:   0%|          | 0/2500 [00:00<?, ?it/s]

d9cf849de83a | epoch 02 | train 0.3056 | val 0.3001 | micro 0.6224 | macro 0.4958 | threshold 0.30


d9cf849de83a | epoch 3:   0%|          | 0/2500 [00:00<?, ?it/s]

d9cf849de83a | epoch 03 | train 0.2824 | val 0.2939 | micro 0.6309 | macro 0.5174 | threshold 0.30


... (4 outputs left out)

,run_id,implementation_version,head,model_name,seed,preprocessing,max_len,dropout,lr,weight_decay,...,loss,micro_f1,macro_f1,hamming_loss,ap_micro,ap_macro,threshold,best_epoch,epochs_run,training_seconds
0,d9cf849de83a,2,cls_dense_tanh_dropout_linear,monologg/koelectra-base-v3-discriminator,42,repeat,128,0.1,0.00002,0.01,...,bce,0.63345,0.515984,0.137705,0.688938,0.552962,0.35,5,5,805.632148


,epoch,train_loss,val_loss,threshold,micro_f1,macro_f1,hamming_loss,precision_micro,recall_micro,predicted_labels_per_sample
0,1,0.396041,0.318516,0.30,0.596107,0.440784,0.156477,0.553790,0.645427,9.1746
1,2,0.305569,0.300115,0.30,0.622403,0.495828,0.147323,0.574761,0.678659,9.2950
2,3,0.282430,0.293881,0.30,0.630923,0.517376,0.145809,0.576566,0.696596,9.5108
3,4,0.268565,0.294310,0.35,0.632986,0.514119,0.139018,0.599791,0.670071,8.7944
4,5,0.260146,0.294383,0.35,0.633450,0.515984,0.137705,0.604703,0.665066,8.6578


성능이 낮은 감정


,label_id,label,support,predicted,precision,recall,f1
30,30,죄책감,89,0,0.000000,0.000000,0.000000
17,17,부끄러움,286,2,0.500000,0.003497,0.006944
18,18,공포/무서움,134,20,0.300000,0.044776,0.077922
26,26,귀찮음,312,53,0.433962,0.073718,0.126027
25,25,패배/자기혐오,183,43,0.418605,0.098361,0.159292
35,35,부담/안_내킴,578,206,0.412621,0.147059,0.216837
36,36,서러움,234,128,0.335938,0.183761,0.237569
19,19,절망,455,306,0.434641,0.292308,0.349540
27,27,힘듦/지침,477,324,0.453704,0.308176,0.367041
9,9,우쭐댐/무시함,772,600,0.470000,0.365285,0.411079


## 기준 모델의 낮은 성능 감정 확인

In [ ]:
run_dir = ARTIFACT_DIR / "runs" / baseline_result["run_id"]

with np.load(run_dir / "validation_predictions.npz") as data:
    probabilities = data["probabilities"]
    labels = data["labels"]

rows = []

for label_id in baseline_labels.sort_values("f1").head(5)["label_id"].astype(int):
    target = labels[:, label_id]
    scores = probabilities[:, label_id]

    positive = target == 1
    negative = ~positive

    grid = np.round(np.arange(0.01, 1.00, 0.01), 2)
    f1_values = [
        f1_score(
            target,
            scores >= threshold,
            zero_division=0,
        )
        for threshold in grid
    ]
    best_index = int(np.argmax(f1_values))

    rows.append({
        "label": LABEL_NAMES[label_id],
        "train_positive": int(y_train[:, label_id].sum()),
        "val_positive": int(target.sum()),
        "max_score": float(scores.max()),
        "positive_mean_score": float(scores[positive].mean()),
        "negative_mean_score": float(scores[negative].mean()),
        "AP": float(average_precision_score(target, scores)),
        "positive_rate": float(target.mean()),
        "best_label_threshold": float(grid[best_index]),
        "best_label_f1": float(f1_values[best_index]),
    })

display(pd.DataFrame(rows).T)

,0,1,2,3,4
label,죄책감,부끄러움,공포/무서움,귀찮음,패배/자기혐오
train_positive,698,2329,1194,2428,1581
val_positive,89,286,134,312,183
max_score,0.26588,0.463748,0.565119,0.652352,0.636466
positive_mean_score,0.069761,0.100471,0.125438,0.138683,0.130404
negative_mean_score,0.017919,0.054257,0.030459,0.054182,0.034966
AP,0.1404,0.143177,0.177688,0.220084,0.193105
positive_rate,0.0178,0.0572,0.0268,0.0624,0.0366
best_label_threshold,0.12,0.09,0.16,0.13,0.16
best_label_f1,0.256983,0.199413,0.28115,0.279955,0.273171


## 학습률 비교

In [ ]:
CURRENT, lr_results = compare_candidates(
    CURRENT,
    [
        {"lr": 1e-5},
        {"lr": 3e-5},
    ],
    stage="01_learning_rate",
)

저장된 결과 사용: d9cf849de83a


34bc1bf978c5 | epoch 1:   0%|          | 0/2500 [00:00<?, ?it/s]

34bc1bf978c5 | epoch 01 | train 0.4215 | val 0.3303 | micro 0.5812 | macro 0.4232 | threshold 0.25


34bc1bf978c5 | epoch 2:   0%|          | 0/2500 [00:00<?, ?it/s]

34bc1bf978c5 | epoch 02 | train 0.3206 | val 0.3110 | micro 0.6078 | macro 0.4615 | threshold 0.30


34bc1bf978c5 | epoch 3:   0%|          | 0/2500 [00:00<?, ?it/s]

... (12 outputs left out)

f387561a6124 | epoch 04 | train 0.2609 | val 0.2935 | micro 0.6350 | macro 0.5218 | threshold 0.35


f387561a6124 | epoch 5:   0%|          | 0/2500 [00:00<?, ?it/s]

f387561a6124 | epoch 05 | train 0.2503 | val 0.2948 | micro 0.6338 | macro 0.5227 | threshold 0.35

Stage: 01_learning_rate


,run_id,implementation_version,head,model_name,seed,preprocessing,max_len,dropout,lr,weight_decay,...,loss,micro_f1,macro_f1,hamming_loss,ap_micro,ap_macro,threshold,best_epoch,epochs_run,training_seconds
0,f387561a6124,2,cls_dense_tanh_dropout_linear,monologg/koelectra-base-v3-discriminator,42,repeat,128,0.1,0.00003,0.01,...,bce,0.634968,0.521813,0.138641,0.692576,0.558613,0.35,4,5,808.255449
1,d9cf849de83a,2,cls_dense_tanh_dropout_linear,monologg/koelectra-base-v3-discriminator,42,repeat,128,0.1,0.00002,0.01,...,bce,0.633450,0.515984,0.137705,0.688938,0.552962,0.35,5,5,805.632148
2,34bc1bf978c5,2,cls_dense_tanh_dropout_linear,monologg/koelectra-base-v3-discriminator,42,repeat,128,0.1,0.00001,0.01,...,bce,0.624624,0.504985,0.151350,0.674683,0.534917,0.30,4,5,817.202483


## 학습 epoch, dropout 비교

In [ ]:
CURRENT, epoch_results = compare_candidates(
    CURRENT,
    [
        {"max_epochs": 8},
    ],
    stage="02_epoch_budget",
)

CURRENT, dropout_results = compare_candidates(
    CURRENT,
    [
        {"dropout": 0.3},
    ],
    stage="03_dropout",
)

저장된 결과 사용: f387561a6124


1c6e2f017248 | epoch 1:   0%|          | 0/2500 [00:00<?, ?it/s]

1c6e2f017248 | epoch 01 | train 0.3982 | val 0.3204 | micro 0.5921 | macro 0.4450 | threshold 0.30


1c6e2f017248 | epoch 2:   0%|          | 0/2500 [00:00<?, ?it/s]

1c6e2f017248 | epoch 02 | train 0.3039 | val 0.2982 | micro 0.6251 | macro 0.5052 | threshold 0.30


1c6e2f017248 | epoch 3:   0%|          | 0/2500 [00:00<?, ?it/s]

... (24 outputs left out)

495dd8911426 | epoch 07 | train 0.2276 | val 0.3014 | micro 0.6356 | macro 0.5358 | threshold 0.35


495dd8911426 | epoch 8:   0%|          | 0/2500 [00:00<?, ?it/s]

495dd8911426 | epoch 08 | train 0.2210 | val 0.3028 | micro 0.6339 | macro 0.5349 | threshold 0.35
조기 종료

Stage: 03_dropout


,run_id,implementation_version,head,model_name,seed,preprocessing,max_len,dropout,lr,weight_decay,...,loss,micro_f1,macro_f1,hamming_loss,ap_micro,ap_macro,threshold,best_epoch,epochs_run,training_seconds
0,495dd8911426,2,cls_dense_tanh_dropout_linear,monologg/koelectra-base-v3-discriminator,42,repeat,128,0.3,0.00003,0.01,...,bce,0.636463,0.540472,0.1425,0.690601,0.559727,0.3,6,8,1293.198639
1,1c6e2f017248,2,cls_dense_tanh_dropout_linear,monologg/koelectra-base-v3-discriminator,42,repeat,128,0.1,0.00003,0.01,...,bce,0.635985,0.538888,0.1423,0.692023,0.563333,0.3,5,7,1125.498044


## Batch size 비교

In [ ]:
CURRENT, batch_results = compare_candidates(
    CURRENT,
    [
        {"batch_size": 32},
    ],
    stage="04_batch_size",
)

save_json(
    CURRENT,
    ARTIFACT_DIR / "current_after_training.json",
)


저장된 결과 사용: 495dd8911426


ade213158a70 | epoch 1:   0%|          | 0/1250 [00:00<?, ?it/s]

ade213158a70 | epoch 01 | train 0.4231 | val 0.3252 | micro 0.5900 | macro 0.4377 | threshold 0.25


ade213158a70 | epoch 2:   0%|          | 0/1250 [00:00<?, ?it/s]

ade213158a70 | epoch 02 | train 0.3141 | val 0.3025 | micro 0.6201 | macro 0.4880 | threshold 0.30


ade213158a70 | epoch 3:   0%|          | 0/1250 [00:00<?, ?it/s]

... (4 outputs left out)

ade213158a70 | epoch 05 | train 0.2576 | val 0.2959 | micro 0.6334 | macro 0.5278 | threshold 0.30


ade213158a70 | epoch 6:   0%|          | 0/1250 [00:00<?, ?it/s]

ade213158a70 | epoch 06 | train 0.2483 | val 0.2963 | micro 0.6335 | macro 0.5232 | threshold 0.35
조기 종료

Stage: 04_batch_size


,run_id,implementation_version,head,model_name,seed,preprocessing,max_len,dropout,lr,weight_decay,...,loss,micro_f1,macro_f1,hamming_loss,ap_micro,ap_macro,threshold,best_epoch,epochs_run,training_seconds
0,495dd8911426,2,cls_dense_tanh_dropout_linear,monologg/koelectra-base-v3-discriminator,42,repeat,128,0.3,0.00003,0.01,...,bce,0.636463,0.540472,0.142500,0.690601,0.559727,0.30,6,8,1293.198639
1,ade213158a70,2,cls_dense_tanh_dropout_linear,monologg/koelectra-base-v3-discriminator,42,repeat,128,0.3,0.00003,0.01,...,bce,0.634520,0.512718,0.137841,0.691004,0.556349,0.35,4,6,492.849360


## 전처리 비교

In [ ]:
CURRENT, preprocessing_results = compare_candidates(
    CURRENT,
    [
        {"preprocessing": "minimal"},
        {"preprocessing": "repeat_symbols"},
    ],
    stage="05_preprocessing",
)

# 실제 전처리 결과가 같은지도 확인
repeat_train, repeat_val = get_texts("repeat")
symbols_train, symbols_val = get_texts("repeat_symbols")

print(
    "repeat와 repeat_symbols가 다른 학습 문장 수:",
    sum(
        left != right
        for left, right in zip(repeat_train, symbols_train)
    ),
)
print(
    "repeat와 repeat_symbols가 다른 검증 문장 수:",
    sum(
        left != right
        for left, right in zip(repeat_val, symbols_val)
    ),
)

저장된 결과 사용: 495dd8911426


cfc55eb597e1 | epoch 1:   0%|          | 0/2500 [00:00<?, ?it/s]

cfc55eb597e1 | epoch 01 | train 0.4016 | val 0.3194 | micro 0.5938 | macro 0.4417 | threshold 0.30


cfc55eb597e1 | epoch 2:   0%|          | 0/2500 [00:00<?, ?it/s]

cfc55eb597e1 | epoch 02 | train 0.3065 | val 0.2993 | micro 0.6232 | macro 0.5012 | threshold 0.30


cfc55eb597e1 | epoch 3:   0%|          | 0/2500 [00:00<?, ?it/s]

... (25 outputs left out)

4aec382c630c | epoch 8:   0%|          | 0/2500 [00:00<?, ?it/s]

4aec382c630c | epoch 08 | train 0.2215 | val 0.3045 | micro 0.6296 | macro 0.5300 | threshold 0.35
조기 종료

Stage: 05_preprocessing


,run_id,implementation_version,head,model_name,seed,preprocessing,max_len,dropout,lr,weight_decay,...,loss,micro_f1,macro_f1,hamming_loss,ap_micro,ap_macro,threshold,best_epoch,epochs_run,training_seconds
0,495dd8911426,2,cls_dense_tanh_dropout_linear,monologg/koelectra-base-v3-discriminator,42,repeat,128,0.3,0.00003,0.01,...,bce,0.636463,0.540472,0.142500,0.690601,0.559727,0.30,6,8,1293.198639
1,cfc55eb597e1,2,cls_dense_tanh_dropout_linear,monologg/koelectra-base-v3-discriminator,42,minimal,128,0.3,0.00003,0.01,...,bce,0.635280,0.537366,0.143177,0.690297,0.559948,0.30,6,8,1307.038745
2,4aec382c630c,2,cls_dense_tanh_dropout_linear,monologg/koelectra-base-v3-discriminator,42,repeat_symbols,128,0.3,0.00003,0.01,...,bce,0.632150,0.527118,0.136895,0.687981,0.556437,0.35,6,8,1305.961339


repeat와 repeat_symbols가 다른 학습 문장 수: 25198
repeat와 repeat_symbols가 다른 검증 문장 수: 3180


## 입력 길이 확인 및 조건부 비교

In [ ]:
selected_train_texts, selected_val_texts = get_texts(
    CURRENT["preprocessing"]
)

selected_stats = {
    "train": input_statistics(
        selected_train_texts, CURRENT["max_len"]
    ),
    "validation": input_statistics(
        selected_val_texts, CURRENT["max_len"]
    ),
}

display(pd.DataFrame(selected_stats).T)

save_json(
    selected_stats,
    ARTIFACT_DIR / "selected_input_statistics.json",
)

if selected_stats["train"]["truncation_rate"] >= 0.01:
    CURRENT, length_results = compare_candidates(
        CURRENT,
        [{"max_len": 192}],
        stage="06_input_length",
    )
else:
    length_results = []
    print("잘리는 학습 문장이 1% 미만이므로 길이 비교 생략")

,mean_tokens,p95,p99,truncation_rate,unk_ratio
train,34.013625,81.0,137.00,0.013225,0.001363
validation,34.786600,82.0,134.01,0.012000,0.001271


저장된 결과 사용: 495dd8911426


587ab5ad8320 | epoch 1:   0%|          | 0/2500 [00:00<?, ?it/s]

587ab5ad8320 | epoch 01 | train 0.4015 | val 0.3202 | micro 0.5935 | macro 0.4454 | threshold 0.30


587ab5ad8320 | epoch 2:   0%|          | 0/2500 [00:00<?, ?it/s]

587ab5ad8320 | epoch 02 | train 0.3065 | val 0.2985 | micro 0.6251 | macro 0.5008 | threshold 0.30


... (7 outputs left out)

587ab5ad8320 | epoch 06 | train 0.2358 | val 0.2992 | micro 0.6333 | macro 0.5401 | threshold 0.30


587ab5ad8320 | epoch 7:   0%|          | 0/2500 [00:00<?, ?it/s]

587ab5ad8320 | epoch 07 | train 0.2264 | val 0.3011 | micro 0.6353 | macro 0.5360 | threshold 0.35
조기 종료

Stage: 06_input_length


,run_id,implementation_version,head,model_name,seed,preprocessing,max_len,dropout,lr,weight_decay,...,loss,micro_f1,macro_f1,hamming_loss,ap_micro,ap_macro,threshold,best_epoch,epochs_run,training_seconds
0,495dd8911426,2,cls_dense_tanh_dropout_linear,monologg/koelectra-base-v3-discriminator,42,repeat,128,0.3,0.00003,0.01,...,bce,0.636463,0.540472,0.1425,0.690601,0.559727,0.30,6,8,1293.198639
1,587ab5ad8320,2,cls_dense_tanh_dropout_linear,monologg/koelectra-base-v3-discriminator,42,repeat,192,0.3,0.00003,0.01,...,bce,0.636403,0.530634,0.1362,0.694608,0.565886,0.35,5,7,1182.124552


## 손실 함수 비교 및 최종 설정 선택

In [ ]:
CURRENT, loss_results = compare_candidates(
    CURRENT,
    [
        {"loss": "weighted_bce"},
    ],
    stage="07_loss",
)

save_json(
    CURRENT,
    ARTIFACT_DIR / "final_config.json",
)

final_result = run_experiment(CURRENT)

print("최종 선택 설정")
display(pd.Series(CURRENT))
display(result_table([baseline_result, final_result]))

저장된 결과 사용: 495dd8911426


7cbe4cad66d4 | epoch 1:   0%|          | 0/2500 [00:00<?, ?it/s]

7cbe4cad66d4 | epoch 01 | train 0.6010 | val 0.5073 | micro 0.5908 | macro 0.4857 | threshold 0.50


7cbe4cad66d4 | epoch 2:   0%|          | 0/2500 [00:00<?, ?it/s]

7cbe4cad66d4 | epoch 02 | train 0.4861 | val 0.4775 | micro 0.6150 | macro 0.5284 | threshold 0.50


7cbe4cad66d4 | epoch 3:   0%|          | 0/2500 [00:00<?, ?it/s]

... (7 outputs left out)

,run_id,implementation_version,head,model_name,seed,preprocessing,max_len,dropout,lr,weight_decay,...,loss,micro_f1,macro_f1,hamming_loss,ap_micro,ap_macro,threshold,best_epoch,epochs_run,training_seconds
0,495dd8911426,2,cls_dense_tanh_dropout_linear,monologg/koelectra-base-v3-discriminator,42,repeat,128,0.3,0.00003,0.01,...,bce,0.636463,0.540472,0.142500,0.690601,0.559727,0.3,6,8,1293.198639
1,7cbe4cad66d4,2,cls_dense_tanh_dropout_linear,monologg/koelectra-base-v3-discriminator,42,repeat,128,0.3,0.00003,0.01,...,weighted_bce,0.626430,0.548401,0.146918,0.678216,0.563299,0.5,4,6,982.724035


저장된 결과 사용: 495dd8911426
최종 선택 설정


,0
implementation_version,2
head,cls_dense_tanh_dropout_linear
model_name,monologg/koelectra-base-v3-discriminator
seed,42
preprocessing,repeat
max_len,128
dropout,0.3
lr,0.00003
weight_decay,0.01
batch_size,16


,run_id,implementation_version,head,model_name,seed,preprocessing,max_len,dropout,lr,weight_decay,...,loss,micro_f1,macro_f1,hamming_loss,ap_micro,ap_macro,threshold,best_epoch,epochs_run,training_seconds
0,495dd8911426,2,cls_dense_tanh_dropout_linear,monologg/koelectra-base-v3-discriminator,42,repeat,128,0.3,0.00003,0.01,...,bce,0.636463,0.540472,0.142500,0.690601,0.559727,0.30,6,8,1293.198639
1,d9cf849de83a,2,cls_dense_tanh_dropout_linear,monologg/koelectra-base-v3-discriminator,42,repeat,128,0.1,0.00002,0.01,...,bce,0.633450,0.515984,0.137705,0.688938,0.552962,0.35,5,5,805.632148


## 공통, 감정별 임계값 비교

In [ ]:
def refine_thresholds(result):
    run_dir = (
        ARTIFACT_DIR / "runs" / result["run_id"]
    )

    with np.load(
        run_dir / "validation_predictions.npz"
    ) as data:
        probabilities = data["probabilities"]
        labels = data["labels"]

    fine_grid = np.round(
        np.arange(0.01, 1.00, 0.01), 2
    )

    table, best = threshold_search(
        labels, probabilities, fine_grid
    )

    table.to_csv(
        run_dir / "fine_thresholds.csv",
        index=False,
    )

    global_summary = {
        **best,
        "ap_micro": result["ap_micro"],
        "ap_macro": result["ap_macro"],
    }

    save_json(
        global_summary,
        run_dir / "fine_global_summary.json",
    )

    labelwise_thresholds = np.full(
        NUM_LABELS,
        best["threshold"],
        dtype=np.float32,
    )

    for label_id in range(NUM_LABELS):
        # 검증셋에 양성 사례가 없는 감정은 공통 임계값 유지
        if labels[:, label_id].sum() == 0:
            continue

        scores = [
            f1_score(
                labels[:, label_id],
                probabilities[:, label_id] >= threshold,
                zero_division=0,
            )
            for threshold in fine_grid
        ]

        labelwise_thresholds[label_id] = fine_grid[
            int(np.argmax(scores))
        ]

    labelwise_metrics = classification_metrics(
        labels, probabilities, labelwise_thresholds
    )

    pd.DataFrame({
        "label": LABEL_NAMES,
        "threshold": labelwise_thresholds,
    }).to_csv(
        run_dir / "labelwise_thresholds.csv",
        index=False,
        encoding="utf-8-sig",
    )

    comparison = pd.DataFrame([
        {
            "method": "coarse_global",
            **classification_metrics(
                labels,
                probabilities,
                result["threshold"],
            ),
        },
        {
            "method": "fine_global",
            **classification_metrics(
                labels,
                probabilities,
                best["threshold"],
            ),
        },
        {
            "method": "labelwise",
            **labelwise_metrics,
        },
    ])

    comparison.to_csv(
        run_dir / "threshold_method_comparison.csv",
        index=False,
    )

    return global_summary, comparison


fine_summary, threshold_comparison = refine_thresholds(
    final_result
)

print("세밀한 공통 임계값:", fine_summary["threshold"])
display(threshold_comparison)
# 이후 웹 추론 및 테스트 평가에 사용할 전역 임계값과 아티팩트 위치
save_json({
    "model_name": MODEL_NAME,
    "run_id": final_result["run_id"],
    "config": CURRENT,
    "global_threshold": fine_summary["threshold"],
    "checkpoint": str(ARTIFACT_DIR / "runs" / final_result["run_id"] / "best.pt"),
    "label_names": LABEL_NAMES,
    "note": "검증셋에서 선택한 값이며 테스트 결과가 아닙니다.",
}, ARTIFACT_DIR / "selected_model.json")


세밀한 공통 임계값: 0.33


,method,micro_f1,macro_f1,hamming_loss,precision_micro,recall_micro,predicted_labels_per_sample
0,coarse_global,0.636463,0.540472,0.142500,0.585438,0.697231,9.3752
1,fine_global,0.637282,0.536037,0.137764,0.602403,0.676448,8.8396
2,labelwise,0.629177,0.557240,0.151186,0.560584,0.716895,10.0670


## 시드 3개 반복 실험

In [ ]:
RUN_REPEATS = True
if RUN_REPEATS:
    repeat_rows = []

    for setting, template in [
        ("baseline", BASE_CONFIG),
        ("final", CURRENT),
    ]:
        for seed in [42, 43, 44]:
            config = copy.deepcopy(template)
            config["seed"] = seed

            result = run_experiment(config)
            fine, _ = refine_thresholds(result)

            repeat_rows.append({
                "setting": setting,
                "seed": seed,
                "run_id": result["run_id"],
                "micro_f1": fine["micro_f1"],
                "macro_f1": fine["macro_f1"],
                "hamming_loss": fine["hamming_loss"],
                "ap_micro": result["ap_micro"],
                "ap_macro": result["ap_macro"],
                "threshold": fine["threshold"],
                "training_seconds": result["training_seconds"],
                "gpu": result["gpu"],
            })

    repeat_table = pd.DataFrame(repeat_rows)

    repeat_summary = (
        repeat_table.groupby("setting")[
            [
                "micro_f1",
                "macro_f1",
                "hamming_loss",
                "ap_micro",
                "ap_macro",
                "training_seconds",
            ]
        ]
        .agg(["mean", "std"])
    )

    display(repeat_table)
    display(repeat_summary)

    repeat_table.to_csv(
        ARTIFACT_DIR / "repeat_results.csv",
        index=False,
        encoding="utf-8-sig",
    )

    repeat_summary.to_csv(
        ARTIFACT_DIR / "repeat_summary.csv",
        encoding="utf-8-sig",
    )


저장된 결과 사용: d9cf849de83a


ae65b8a94fa6 | epoch 1:   0%|          | 0/2500 [00:00<?, ?it/s]

ae65b8a94fa6 | epoch 01 | train 0.3966 | val 0.3189 | micro 0.5950 | macro 0.4400 | threshold 0.30


ae65b8a94fa6 | epoch 2:   0%|          | 0/2500 [00:00<?, ?it/s]

ae65b8a94fa6 | epoch 02 | train 0.3052 | val 0.3003 | micro 0.6209 | macro 0.4961 | threshold 0.30


ae65b8a94fa6 | epoch 3:   0%|          | 0/2500 [00:00<?, ?it/s]

... (43 outputs left out)

ef35405231d2 | epoch 8:   0%|          | 0/2500 [00:00<?, ?it/s]

ef35405231d2 | epoch 08 | train 0.2217 | val 0.3033 | micro 0.6320 | macro 0.5302 | threshold 0.35
조기 종료


,setting,seed,run_id,micro_f1,macro_f1,hamming_loss,ap_micro,ap_macro,threshold,training_seconds,gpu
0,baseline,42,d9cf849de83a,0.633751,0.520692,0.140736,0.688938,0.552962,0.33,805.632148,NVIDIA A100-SXM4-40GB
1,baseline,43,ae65b8a94fa6,0.634687,0.513482,0.136905,0.689330,0.553785,0.36,815.447839,NVIDIA A100-SXM4-40GB
2,baseline,44,27d5e661f438,0.632564,0.515575,0.139186,0.690483,0.554555,0.34,821.576166,NVIDIA A100-SXM4-40GB
3,final,42,495dd8911426,0.637282,0.536037,0.137764,0.690601,0.559727,0.33,1293.198639,NVIDIA A100-SXM4-40GB
4,final,43,cf279d7160aa,0.635147,0.529362,0.135432,0.690091,0.562366,0.35,1146.847428,NVIDIA A100-SXM4-40GB
5,final,44,ef35405231d2,0.635214,0.534141,0.138473,0.689903,0.559262,0.35,1309.382066,NVIDIA A100-SXM4-40GB


micro_f1            macro_f1           hamming_loss            \
              mean       std      mean       std         mean       std   
setting                                                                   
baseline  0.633667  0.001064  0.516583  0.003709     0.138942  0.001928   
final     0.635881  0.001214  0.533180  0.003439     0.137223  0.001591   

          ap_micro            ap_macro           training_seconds             
              mean       std      mean       std             mean        std  
setting                                                                       
baseline  0.689584  0.000803  0.553767  0.000797       814.218718   8.042759  
final     0.690198  0.000361  0.560452  0.001674      1249.809377  89.534061

## 전체 실험 및 감정별 결과 정리

In [ ]:
all_results = []

for path in sorted(
    (ARTIFACT_DIR / "runs").glob("*/summary.json")
):
    all_results.append(
        json.loads(path.read_text(encoding="utf-8"))
    )

all_experiments = result_table(all_results)

display(all_experiments)

all_experiments.to_csv(
    ARTIFACT_DIR / "all_experiments.csv",
    index=False,
    encoding="utf-8-sig",
)


def load_label_results(result):
    run_dir = (
        ARTIFACT_DIR / "runs" / result["run_id"]
    )

    with np.load(
        run_dir / "validation_predictions.npz"
    ) as data:
        probabilities = data["probabilities"]
        labels = data["labels"]

    fine = json.loads(
        (run_dir / "fine_global_summary.json").read_text(
            encoding="utf-8"
        )
    )

    return labelwise_table(
        labels, probabilities, fine["threshold"]
    )


baseline_label_table = load_label_results(
    baseline_result
)
final_label_table = load_label_results(
    final_result
)

label_comparison = final_label_table.merge(
    baseline_label_table[["label_id", "f1"]].rename(
        columns={"f1": "baseline_f1"}
    ),
    on="label_id",
)

label_comparison["f1_change"] = (
    label_comparison["f1"]
    - label_comparison["baseline_f1"]
)

print("최종 성능이 낮은 감정")
display(
    label_comparison.sort_values("f1").head(10)
)

print("개선 폭이 큰 감정")
display(
    label_comparison.sort_values(
        "f1_change", ascending=False
    ).head(10)
)

label_comparison.to_csv(
    ARTIFACT_DIR / "label_comparison.csv",
    index=False,
    encoding="utf-8-sig",
)

print("최종 모델 checkpoint:")
print(
    ARTIFACT_DIR / "runs"
    / final_result["run_id"] / "best.pt"
)

,run_id,implementation_version,head,model_name,seed,preprocessing,max_len,dropout,lr,weight_decay,...,loss,micro_f1,macro_f1,hamming_loss,ap_micro,ap_macro,threshold,best_epoch,epochs_run,training_seconds
0,495dd8911426,2,cls_dense_tanh_dropout_linear,monologg/koelectra-base-v3-discriminator,42,repeat,128,0.3,0.00003,0.01,...,bce,0.636463,0.540472,0.142500,0.690601,0.559727,0.30,6,8,1293.198639
1,587ab5ad8320,2,cls_dense_tanh_dropout_linear,monologg/koelectra-base-v3-discriminator,42,repeat,192,0.3,0.00003,0.01,...,bce,0.636403,0.530634,0.136200,0.694608,0.565886,0.35,5,7,1182.124552
2,1c6e2f017248,2,cls_dense_tanh_dropout_linear,monologg/koelectra-base-v3-discriminator,42,repeat,128,0.1,0.00003,0.01,...,bce,0.635985,0.538888,0.142300,0.692023,0.563333,0.30,5,7,1125.498044
3,cfc55eb597e1,2,cls_dense_tanh_dropout_linear,monologg/koelectra-base-v3-discriminator,42,minimal,128,0.3,0.00003,0.01,...,bce,0.635280,0.537366,0.143177,0.690297,0.559948,0.30,6,8,1307.038745
4,ef35405231d2,2,cls_dense_tanh_dropout_linear,monologg/koelectra-base-v3-discriminator,44,repeat,128,0.3,0.00003,0.01,...,bce,0.635214,0.534141,0.138473,0.689903,0.559262,0.35,6,8,1309.382066
5,cf279d7160aa,2,cls_dense_tanh_dropout_linear,monologg/koelectra-base-v3-discriminator,43,repeat,128,0.3,0.00003,0.01,...,bce,0.635147,0.529362,0.135432,0.690091,0.562366,0.35,5,7,1146.847428
6,f387561a6124,2,cls_dense_tanh_dropout_linear,monologg/koelectra-base-v3-discriminator,42,repeat,128,0.1,0.00003,0.01,...,bce,0.634968,0.521813,0.138641,0.692576,0.558613,0.35,4,5,808.255449
7,ae65b8a94fa6,2,cls_dense_tanh_dropout_linear,monologg/koelectra-base-v3-discriminator,43,repeat,128,0.1,0.00002,0.01,...,bce,0.634604,0.515454,0.138550,0.689330,0.553785,0.35,4,5,815.447839
8,ade213158a70,2,cls_dense_tanh_dropout_linear,monologg/koelectra-base-v3-discriminator,42,repeat,128,0.3,0.00003,0.01,...,bce,0.634520,0.512718,0.137841,0.691004,0.556349,0.35,4,6,492.849360
9,d9cf849de83a,2,cls_dense_tanh_dropout_linear,monologg/koelectra-base-v3-discriminator,42,repeat,128,0.1,0.00002,0.01,...,bce,0.633450,0.515984,0.137705,0.688938,0.552962,0.35,5,5,805.632148


최종 성능이 낮은 감정


,label_id,label,support,predicted,precision,recall,f1,baseline_f1,f1_change
17,17,부끄러움,286,24,0.291667,0.024476,0.045161,0.006920,0.038241
30,30,죄책감,89,6,0.500000,0.033708,0.063158,0.000000,0.063158
25,25,패배/자기혐오,183,73,0.383562,0.153005,0.218750,0.172414,0.046336
26,26,귀찮음,312,156,0.358974,0.179487,0.239316,0.142857,0.096459
18,18,공포/무서움,134,87,0.344828,0.223881,0.271493,0.112500,0.158993
36,36,서러움,234,184,0.326087,0.256410,0.287081,0.250000,0.037081
35,35,부담/안_내킴,578,436,0.355505,0.268166,0.305720,0.242718,0.063001
19,19,절망,455,440,0.397727,0.384615,0.391061,0.366834,0.024227
27,27,힘듦/지침,477,432,0.437500,0.396226,0.415842,0.382212,0.033630
9,9,우쭐댐/무시함,772,636,0.463836,0.382124,0.419034,0.424495,-0.005461


개선 폭이 큰 감정


,label_id,label,support,predicted,precision,recall,f1,baseline_f1,f1_change
18,18,공포/무서움,134,87,0.344828,0.223881,0.271493,0.112500,0.158993
26,26,귀찮음,312,156,0.358974,0.179487,0.239316,0.142857,0.096459
30,30,죄책감,89,6,0.500000,0.033708,0.063158,0.000000,0.063158
35,35,부담/안_내킴,578,436,0.355505,0.268166,0.305720,0.242718,0.063001
25,25,패배/자기혐오,183,73,0.383562,0.153005,0.218750,0.172414,0.046336
17,17,부끄러움,286,24,0.291667,0.024476,0.045161,0.006920,0.038241
36,36,서러움,234,184,0.326087,0.256410,0.287081,0.250000,0.037081
27,27,힘듦/지침,477,432,0.437500,0.396226,0.415842,0.382212,0.033630
19,19,절망,455,440,0.397727,0.384615,0.391061,0.366834,0.024227
11,11,비장함,399,352,0.502841,0.443609,0.471372,0.448598,0.022773


최종 모델 checkpoint:
/content/drive/MyDrive/26_KOTE/artifacts/week11_koelectra/runs/495dd8911426/best.pt


## 반복 실험 결과 요약

In [ ]:
repeat_path = ARTIFACT_DIR / "repeat_results.csv"
if repeat_path.exists():
    repeats = pd.read_csv(repeat_path)
    display(repeats.groupby("setting")[["micro_f1", "macro_f1", "hamming_loss", "ap_micro"]].agg(["mean", "std"]))
else:
    print("시드 반복 실험이 아직 없습니다. 위 결과는 seed 42의 검증 결과입니다.")

print("결과 폴더:", ARTIFACT_DIR)
print("추론용 설정:", ARTIFACT_DIR / "selected_model.json")
print("최종 테스트 평가는 아직 수행하지 않았습니다.")

micro_f1            macro_f1           hamming_loss            \
              mean       std      mean       std         mean       std   
setting                                                                   
baseline  0.633667  0.001064  0.516583  0.003709     0.138942  0.001928   
final     0.635881  0.001214  0.533180  0.003439     0.137223  0.001591   

          ap_micro            
              mean       std  
setting                       
baseline  0.689584  0.000803  
final     0.690198  0.000361

결과 폴더: /content/drive/MyDrive/26_KOTE/artifacts/week11_koelectra
추론용 설정: /content/drive/MyDrive/26_KOTE/artifacts/week11_koelectra/selected_model.json
최종 테스트 평가는 아직 수행하지 않았습니다.
